In [1]:
import json
import os
import math
import random
import pandas as pd
from collections import defaultdict, Counter
from src.grammar.basic_generator import BasicGrammar
from src.grammar.controlled_generator import ControlledFSM
from src.grammar.utils import transition_entropy, estimate_entropy, estimate_valid_sample_difficulty#, estimate_diversity
from src.grammar.examples import GRAMMARS

In [2]:
fsms = {}
for gname, gjson in GRAMMARS.items():
    fsms[gname] = BasicGrammar.from_json(
        gjson, #.replace("\n", ""),
        is_path=False
    )
    print(gname, "loaded")
for gname in os.listdir("generated_grammars"):
    fsms[gname.split(".")[0]] = BasicGrammar.from_json(f"generated_grammars/{gname}", is_path=True)
    print(gname.split(".")[0], "loaded")

SIMPLE_LOOP_GRAMMAR loaded
ALTERNATING_AB_LOOP_GRAMMAR loaded
ALTERNATING_ABEXPONENT_LOOP_GRAMMAR loaded
ALTERNATING_CONSONANT_VOWEL_LOOP_GRAMMAR loaded
ALTERNATING_CONSONANT_VOWELEXPONENT_LOOP_GRAMMAR loaded
ALTERNATING_GIBBEGIBBERISH_LOOP_GRAMMAR loaded
ALTERNATING_GIBBEGIBBERISHEXPONENT_LOOP_GRAMMAR loaded
LADDER_GRAMMAR loaded
WAYBACK_GRAMMAR loaded
ALTERNATING_MUSTSTOP_LOOP_GRAMMAR loaded
ALTERNATING_MUSTSTOPEXPONENT_LOOP_GRAMMAR loaded
WAYBACK_WORD_GRAMMAR loaded
s11_a3 loaded
s13_a10 loaded
s3_a3 loaded
s3_a4 loaded
s4_a2 loaded
s4_a4 loaded
s4_a6 loaded
s4_a8 loaded
s5_a4 loaded
s5_a5 loaded
s5_a5bis loaded
s6_a4 loaded
s7_a3 loaded
s7_a7 loaded
s9_a9 loaded


In [3]:
NUM_ITERATIONS_MOST_GRAMMARS = 8
to_keep = {
    "ALTERNATING_ABEXPONENT_LOOP_GRAMMAR": 4,
    "ALTERNATING_CONSONANT_VOWELEXPONENT_LOOP_GRAMMAR": 5,
    "LADDER_GRAMMAR": NUM_ITERATIONS_MOST_GRAMMARS,
    "WAYBACK_GRAMMAR": NUM_ITERATIONS_MOST_GRAMMARS,
    "ALTERNATING_MUSTSTOPEXPONENT_LOOP_GRAMMAR": 5,
    "WAYBACK_WORD_GRAMMAR": NUM_ITERATIONS_MOST_GRAMMARS,
    "s11_a3": NUM_ITERATIONS_MOST_GRAMMARS,
    "s13_a10": NUM_ITERATIONS_MOST_GRAMMARS,
    "s3_a3": NUM_ITERATIONS_MOST_GRAMMARS,
    "s3_a4": NUM_ITERATIONS_MOST_GRAMMARS,
    "s4_a2": NUM_ITERATIONS_MOST_GRAMMARS,
    "s4_a4": NUM_ITERATIONS_MOST_GRAMMARS,
    "s4_a8": NUM_ITERATIONS_MOST_GRAMMARS,
    "s5_a4": NUM_ITERATIONS_MOST_GRAMMARS,
    "s5_a5": NUM_ITERATIONS_MOST_GRAMMARS,
    "s5_a5bis": NUM_ITERATIONS_MOST_GRAMMARS,
    "s6_a4": NUM_ITERATIONS_MOST_GRAMMARS,
    "s7_a3": NUM_ITERATIONS_MOST_GRAMMARS,
    "s7_a7": NUM_ITERATIONS_MOST_GRAMMARS,
    "s9_a9": NUM_ITERATIONS_MOST_GRAMMARS,
}

In [4]:
len(to_keep)


20

In [5]:
DIFFICULTIES = {}
max_len_v = 6
for fsmname in to_keep:
    fsm = fsms[fsmname]
    print(fsmname)
    #norm_den = (len(fsm._compute_effective_alphabet()) ** max_len_v)
    DIFFICULTIES[fsmname] = 0.5*(abs(transition_entropy(fsm)) + abs(estimate_entropy(fsm)))# +\
    #math.log(norm_den) * len(fsm.enumerate_valid_samples(max_len=max_len_v))/norm_den

ALTERNATING_ABEXPONENT_LOOP_GRAMMAR
ALTERNATING_CONSONANT_VOWELEXPONENT_LOOP_GRAMMAR
LADDER_GRAMMAR
WAYBACK_GRAMMAR
ALTERNATING_MUSTSTOPEXPONENT_LOOP_GRAMMAR
WAYBACK_WORD_GRAMMAR
s11_a3
s13_a10
s3_a3
s3_a4
s4_a2
s4_a4
s4_a8
s5_a4
s5_a5
s5_a5bis
s6_a4
s7_a3
s7_a7
s9_a9


In [6]:
for fsmname, difficulty in sorted(DIFFICULTIES.items(), key=lambda x: x[1]):
    print(fsmname, difficulty)

ALTERNATING_MUSTSTOPEXPONENT_LOOP_GRAMMAR 0.13435634778684213
s3_a3 0.20890879332266343
s3_a4 0.21058514166025183
s5_a4 0.2686777247967251
ALTERNATING_ABEXPONENT_LOOP_GRAMMAR 0.274653102543649
LADDER_GRAMMAR 0.28165654378154614
s4_a2 0.2830441816585577
s7_a7 0.318948902272073
s4_a8 0.32830434740339764
s5_a5 0.3390702809906492
WAYBACK_WORD_GRAMMAR 0.3614501185273049
s7_a3 0.3963611904976536
s5_a5bis 0.45885318944945075
WAYBACK_GRAMMAR 0.47481674255133477
s4_a4 0.48380163667465087
s11_a3 0.4857068058862633
s6_a4 0.5211495675459927
s13_a10 0.5726670968765674
s9_a9 0.6438235810287926
ALTERNATING_CONSONANT_VOWELEXPONENT_LOOP_GRAMMAR 1.4066777023473211


In [7]:
UPPER_LIMIT_VALID_SAMPLE_GENERATION = 1000
MAX_CHARS = 50
first_valid_samples = {}
for fsmname in fsms.keys():
    first_valid_samples[fsmname] = fsms[fsmname].enumerate_valid_samples(
            max_len=MAX_CHARS,
            early_stop_function=lambda x: len(x) > UPPER_LIMIT_VALID_SAMPLE_GENERATION
        )
    first_valid_samples[fsmname] = [s for s in first_valid_samples[fsmname] if s]

In [8]:
MAX_NUMBER_INVALID = 300
first_invalid_samples = defaultdict(set)
difficulty_invalid = defaultdict(dict)
for fsmname in fsms.keys():
    print(fsmname)
    original_alphabet = fsms[fsmname].alphabet
    if len(original_alphabet) < 3:
        fsms[fsmname].alphabet = list(set(fsms[fsmname].alphabet).union({"a","b","c"}))
    while len(first_invalid_samples[fsmname]) < MAX_NUMBER_INVALID:
        ret = fsms[fsmname].generate_plausible_invalid(
            expected_len=(el:=random.randint(1, MAX_CHARS)), num_disruptions_range=(1, el))
        if ret is None: continue
        invalid_sample, num_disruptions = ret
        first_invalid_samples[fsmname].add(invalid_sample)
        difficulty_invalid[fsmname][invalid_sample] = 1 - num_disruptions / len(invalid_sample)
    first_invalid_samples[fsmname] = sorted(first_invalid_samples[fsmname], key=lambda x: len(x))
    fsms[fsmname].alphabet = original_alphabet

SIMPLE_LOOP_GRAMMAR
ALTERNATING_AB_LOOP_GRAMMAR
ALTERNATING_ABEXPONENT_LOOP_GRAMMAR
ALTERNATING_CONSONANT_VOWEL_LOOP_GRAMMAR
ALTERNATING_CONSONANT_VOWELEXPONENT_LOOP_GRAMMAR
ALTERNATING_GIBBEGIBBERISH_LOOP_GRAMMAR
ALTERNATING_GIBBEGIBBERISHEXPONENT_LOOP_GRAMMAR
LADDER_GRAMMAR
WAYBACK_GRAMMAR
ALTERNATING_MUSTSTOP_LOOP_GRAMMAR
ALTERNATING_MUSTSTOPEXPONENT_LOOP_GRAMMAR
WAYBACK_WORD_GRAMMAR
s11_a3
s13_a10
s3_a3
s3_a4
s4_a2
s4_a4
s4_a6
s4_a8
s5_a4
s5_a5
s5_a5bis
s6_a4
s7_a3
s7_a7
s9_a9


In [9]:
len(first_valid_samples["ALTERNATING_ABEXPONENT_LOOP_GRAMMAR"])

1002

In [10]:
first_valid_samples.keys()

dict_keys(['SIMPLE_LOOP_GRAMMAR', 'ALTERNATING_AB_LOOP_GRAMMAR', 'ALTERNATING_ABEXPONENT_LOOP_GRAMMAR', 'ALTERNATING_CONSONANT_VOWEL_LOOP_GRAMMAR', 'ALTERNATING_CONSONANT_VOWELEXPONENT_LOOP_GRAMMAR', 'ALTERNATING_GIBBEGIBBERISH_LOOP_GRAMMAR', 'ALTERNATING_GIBBEGIBBERISHEXPONENT_LOOP_GRAMMAR', 'LADDER_GRAMMAR', 'WAYBACK_GRAMMAR', 'ALTERNATING_MUSTSTOP_LOOP_GRAMMAR', 'ALTERNATING_MUSTSTOPEXPONENT_LOOP_GRAMMAR', 'WAYBACK_WORD_GRAMMAR', 's11_a3', 's13_a10', 's3_a3', 's3_a4', 's4_a2', 's4_a4', 's4_a6', 's4_a8', 's5_a4', 's5_a5', 's5_a5bis', 's6_a4', 's7_a3', 's7_a7', 's9_a9'])

In [11]:
Counter([len(word) for word in first_valid_samples["s6_a4"]])

Counter({8: 398, 7: 202, 9: 197, 6: 88, 5: 62, 4: 28, 3: 12, 2: 10, 1: 4})

In [12]:
difficulty_invalid

defaultdict(dict,
            {'SIMPLE_LOOP_GRAMMAR': {'cbbacabababbbbacacabcbcbbbcabaab': 0.3125,
              'cacbaa': 0.5,
              'abcbbbaabababacbbcaaaacaabaaacbccacbaacabaaabb': 0.4782608695652174,
              'aaaaaaaabaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaab': 0.9591836734693877,
              'bccbbbbcbbbcbcbcccbaaacbbcabbcbbab': 0.1470588235294118,
              'cccbccbbbccccacbbbcbc': 0.04761904761904767,
              'bbcbbccbacabacaababcacbbbcbbacbba': 0.2727272727272727,
              'aaaaaababbaabaaba': 0.7058823529411764,
              'cabbbbccacabcbccbcbaaa': 0.2727272727272727,
              'aaaaaaaaabaaaaaaaaaaaaaaaabaaaaaaaaaaaabaaaac': 0.9111111111111111,
              'cbcb': 0.0,
              'cbbaccbbabcbbcbcbcccacccbcbccacccbcbbaacc': 0.14634146341463417,
              'aaaacaccaaaacaaaaaaaaacaaaaaaaaaaaa': 0.8571428571428572,
              'accacccbcccbcccbabccabcccbcbcc': 0.1333333333333333,
              'aaaaacaacaaabaabbaaaacaacaaaacaa': 0.

In [13]:
{k: len(v) for k, v in first_valid_samples.items() if k in to_keep}

{'ALTERNATING_ABEXPONENT_LOOP_GRAMMAR': 1002,
 'ALTERNATING_CONSONANT_VOWELEXPONENT_LOOP_GRAMMAR': 1010,
 'LADDER_GRAMMAR': 1001,
 'WAYBACK_GRAMMAR': 1002,
 'ALTERNATING_MUSTSTOPEXPONENT_LOOP_GRAMMAR': 1002,
 'WAYBACK_WORD_GRAMMAR': 1002,
 's11_a3': 1002,
 's13_a10': 1004,
 's3_a3': 1001,
 's3_a4': 1001,
 's4_a2': 1002,
 's4_a4': 1002,
 's4_a8': 1001,
 's5_a4': 1001,
 's5_a5': 1001,
 's5_a5bis': 1001,
 's6_a4': 1001,
 's7_a3': 1001,
 's7_a7': 1001,
 's9_a9': 1006}

In [14]:
#{k: len(v) for k, v in first_invalid_samples.items() if k in to_keep}

In [16]:
df_data = []
columns = [
    "training_valid_samples",
    "training_invalid_samples",
    "inference_samples_labels_difficulty",
    "grammar",
    "grammar_name",
]
MIN_NUM_TRAININGS_VALID = 20
MAX_NUM_TRAININGS_VALID = 100
MIN_NUM_TRAININGS_INVALID = 10
NUM_INFERENCE_SAMPLES = 25
for fsmname, num_datasets in to_keep.items():
    print(fsmname)
    fsm = fsms[fsmname]
    for _ in range(num_datasets):
        num_trainings_valid = random.randint(MIN_NUM_TRAININGS_VALID, MAX_NUM_TRAININGS_VALID)
        num_trainings_invalid = random.randint(MIN_NUM_TRAININGS_INVALID, int(num_trainings_valid * 0.75))
        # we want to favor more valid samples:
        num_inference_valid = random.randint(random.randint(0, NUM_INFERENCE_SAMPLES), NUM_INFERENCE_SAMPLES)
        trainings_valid_samples = random.choices(
            first_valid_samples[fsmname],
            weights=[1/len(s) for s in first_valid_samples[fsmname]], # must have made sure no empty string
            k=num_trainings_valid
        )
        trainings_valid_samples = sorted(trainings_valid_samples, key=lambda x: len(x))
        trainings_invalid_samples = random.choices(
            first_invalid_samples[fsmname],
            weights=[1/len(s) for s in first_invalid_samples[fsmname]], # must have made sure no empty string
            k=num_trainings_invalid
        )
        trainings_invalid_samples = sorted(trainings_invalid_samples, key=lambda x: len(x))
        inference_valid_samples = random.choices(
            [s for s in first_valid_samples[fsmname] if s not in trainings_valid_samples],
            weights=[1/len(s) for s in first_valid_samples[fsmname] if s not in trainings_valid_samples], # must have made sure no empty string
            k=num_inference_valid
        )
        inference_invalid_samples = random.choices(
            [s for s in first_invalid_samples[fsmname] if s not in trainings_invalid_samples],
            weights=[1/len(s) for s in first_invalid_samples[fsmname] if s not in trainings_invalid_samples], # must have made sure no empty string
            k=NUM_INFERENCE_SAMPLES - num_inference_valid
        )
        inference_samples = [[s, True] for s in inference_valid_samples] + [[s, False] for s in inference_invalid_samples]
        for isample in inference_samples:
            if isample[1]:
                isample.append(estimate_valid_sample_difficulty(isample[0], fsm, trainings_valid_samples))
            else:
                isample.append(difficulty_invalid[fsmname][isample[0]])
        random.shuffle(inference_samples)
        df_data.append(
            {
                "training_valid_samples": json.dumps(trainings_valid_samples),
                "training_invalid_samples": json.dumps(trainings_invalid_samples),
                "inference_samples_labels_difficulty": json.dumps(inference_samples),
                "grammar": fsm.to_jsondumps(),
                "grammar_name": fsmname,
            }
        )

ALTERNATING_ABEXPONENT_LOOP_GRAMMAR
ALTERNATING_CONSONANT_VOWELEXPONENT_LOOP_GRAMMAR
LADDER_GRAMMAR
WAYBACK_GRAMMAR
ALTERNATING_MUSTSTOPEXPONENT_LOOP_GRAMMAR
WAYBACK_WORD_GRAMMAR
s11_a3
s13_a10
s3_a3
s3_a4
s4_a2
s4_a4
s4_a8
s5_a4
s5_a5
s5_a5bis
s6_a4
s7_a3
s7_a7
s9_a9


In [17]:
df = pd.DataFrame(df_data, columns=columns)

In [20]:
len(df)

150

In [22]:
df.to_csv("generated_datasets/artificial_grammars.csv", index=None)

In [25]:
df.to_excel("generated_datasets/artificial_grammars.xlsx", index=None)